# Engram: does a small helper model improve memory?

A small open model writes facts linked to the exact sentences when a chat closes, and search phrases for each question. Gemma 3 4B always writes the final answers, so only the helper changes. Development conversations 0 to 4 only. Benchmark data only.

**Colab:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Keep the tab open.

**Kaggle:** Settings > Accelerator > GPU T4, and Settings > Internet > On, then Run All.

If it stops, run all again. It carries on from what was saved.

In [ ]:
ANSWER_MODEL = "gemma3:4b"
HELPERS = ["gemma3:4b", "gemma3:1b"]   # larger first; each one finished is saved
TAG = "dev"
COMMIT = "b9f000e"
import os, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Turn on a T4 GPU in the settings. If none is offered, the free allowance is used up: try again later.")
print("GPU:", gpu)
KAGGLE = os.path.exists("/kaggle/working")
print("Running on", "Kaggle" if KAGGLE else "Colab")

In [ ]:
import os
if KAGGLE:
    SAVE, WORK = "/kaggle/working/engram_results", "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE, WORK = "/content/drive/MyDrive/engram_results", "/content"
os.makedirs(SAVE, exist_ok=True)
print("Results go to", SAVE)

In [ ]:
%cd {WORK}
!rm -rf Engram
!git clone -q https://github.com/manjunathhari/Engram.git
%cd {WORK}/Engram
!git checkout -q {COMMIT}
!git clone -q --depth 1 https://github.com/snap-research/locomo.git
!pip install -q wordllama scikit-learn spacy
!python -m spacy download en_core_web_sm
import os, shutil, wordllama
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
print("tools ready")

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1", OLLAMA_MAX_LOADED_MODELS="2")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
have = []
for m in dict.fromkeys([ANSWER_MODEL] + HELPERS):
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(m, "ready" if r.returncode == 0 else "NOT available, will be skipped")
    if r.returncode == 0: have.append(m)
if ANSWER_MODEL not in have: raise SystemExit("The answer model could not be downloaded.")
HELPERS = [m for m in HELPERS if m in have]

In [ ]:
%%writefile helper_test.py
"""Does a small helper model improve Engram's search? Development conversations 0-4 only. Benchmark data only.
The helper writes facts linked to source sentences when a chat closes, and search phrases for each question.
The exact sentences stay the record. A larger model always writes the final answers, so only the helper changes."""
import argparse, json, math, os, re, sys, tempfile, time
from collections import Counter
import numpy as np
sys.path.insert(0, os.getcwd())
import vps_kit as K
import engram_chat as E
from engram import Config
import engram_parallel as P

ap = argparse.ArgumentParser()
for x in ("url", "answer_model", "tag"): ap.add_argument("--" + x, required=True)
ap.add_argument("--helpers", required=True, help="comma separated")
ap.add_argument("--sample", type=int, default=240)
ap.add_argument("--budget", type=int, default=800)
a = ap.parse_args()
ids = [0, 1, 2, 3, 4]
os.makedirs("results", exist_ok=True)
tools = P._load_tools()
cfg = Config(**json.load(open("final_config.json")))
tmp = tempfile.mkdtemp()
convs = {c: K.load_conv(K.DATA[c]) for c in ids}


def replay(conv, name, search):
    m = E.ChatMemory(f"{tmp}/{name}.db", cfg=cfg, durable=False, search=search)
    last = None
    for t in conv.turns:
        if t["session"] != last:
            if last is not None: m.close_chat(write_note=False, sleep=False)
            m.open_chat(t["date"]); last = t["session"]
        m.add(t["speaker"], t["text"], date=t["date"], ref=t["id"], caption=t["cap"], commit=False)
    m.db.commit(); m.close_chat(write_note=False)
    return m


print("building memories", flush=True)
MEM = {c: replay(convs[c], f"s{c}", P.ChannelSearch("plain", tools=tools)) for c in ids}
CUR = {c: replay(convs[c], f"p{c}", None) for c in ids}
REF = {c: dict(MEM[c].db.execute("SELECT ref, sid FROM sentences").fetchall()) for c in ids}
GOLD = {c: [(q, {REF[c][e] for e in q["ev"] if e in REF[c]}) for q in convs[c].qs if q["cat"] != 5 and q["ev"]] for c in ids}
print("training networks", flush=True)
NET = {}
for c in ids:
    ex = [(MEM[d].search, q["q"], g) for d in ids if d != c for q, g in GOLD[d] if g]
    NET[c] = [P.train(ex, s) for s in (1, 2, 3)]  # three starts averaged, so one lucky start does not decide the result
print("networks ready", flush=True)


def ranks(sids):
    return {s: r for r, s in enumerate(sids)}


def net_order(c, q):
    s = MEM[c].search
    f = s.features(q)
    p = np.mean([m[k][1].predict_proba(m[k][0].transform(f["tables"][k]))[:, 1] for m in NET[c] for k in P.CHANNELS], axis=0)
    first = f["pool"][np.argsort(-p)].tolist()
    have = set(first)
    return [s.sids[i] for i in first + [x for x in np.argsort(-f["plain"]).tolist() if x not in have]]


def fuse(lists):
    """The network's order counts three times as much. Each helper list gives only its top 40, so a poor helper cannot bury good results."""
    score = Counter()
    for n, sids in enumerate(lists):
        short = list(dict.fromkeys(sids)) if n == 0 else list(dict.fromkeys(sids))[:40]
        for r, x in enumerate(short): score[x] += (3 if n == 0 else 1) / (60 + r)
    return [x for x, _ in score.most_common()]


# ---------- the helper ----------
FACT_PROMPT = """Below is one conversation. Each line starts with its id. Write the facts worth remembering later.
Write one line per fact, at most 12, in exactly this form:
ID | fact
The ID must be copied from the line the fact came from. If a fact uses several lines, separate their ids with commas.
Choose the facts from the whole conversation, not only its start. Each fact must make sense alone: use full names instead of he, she or they, and give real dates from the square brackets. Check who actually did each thing. Skip greetings, thanks, compliments and small talk. Use only what is written. Write nothing else.

Conversation:
{chat}

Facts:
"""
CUE_PROMPT = """A question will be used to search someone's past conversations. Write up to 3 short search phrases that could find the answer, using other words someone might have used. If the answer needs separate facts, write one phrase for each fact. Write one phrase per line and nothing else.

Question: {q}

Phrases:
"""


def load(path):
    return json.load(open(path)) if os.path.exists(path) else {}


def overlap(fact_words, sent_words):
    return len(fact_words & sent_words) / max(1, len(fact_words))


def parse_facts(reply, session, sentences):
    """Accepts the forms models really write. Then ordinary code checks each fact against the sentences of this chat:
    a fact goes to the sentence that supports it, and a fact supported by no sentence is dropped."""
    out, seen = [], set()
    prefix = f"D{session}:"
    tokens = {k: set(E.words(v)) for k, v in sentences.items()}
    names = {w for k, v in sentences.items() for w in E.words(v.split(":")[0])}
    for line in reply.splitlines():
        line = line.strip().strip("*-• ")
        cited = [x for x in re.findall(r"D\d+:\d+", line) if x.startswith(prefix) and x in sentences]
        text = re.sub(r"\s*,?\s*D\d+:\d+\s*,?", " ", line)
        text = re.sub(r"^\s*\d{1,2}:\d{2}(\s*[ap]m)?( on [^|]*)?\|?", "", text, flags=re.I)
        if "|" in text:
            left, right = text.split("|", 1)
            if not cited and re.fullmatch(r"[\s\d,]*", left):
                cited = [prefix + x for x in re.findall(r"\d+", left) if prefix + x in sentences]
            text = right if len(right.strip()) > 8 else left
        elif not cited:
            continue
        text = re.sub(r"^[\s:\-|]+", "", text).strip()
        if len(text) <= 8: continue
        fw = set(E.words(text)) - names
        if not fw: continue
        best = max(sentences, key=lambda k: overlap(fw, tokens[k]))
        score = overlap(fw, tokens[best])
        if score < .2: continue
        turns = [t for t in cited if overlap(fw, tokens[t]) >= .2] or [best]
        key = text.lower()
        if key in seen: continue
        seen.add(key)
        out.append(dict(fact=text[:300], turns=turns, cited=cited, moved=turns != cited))
    return out[:12]


def write_facts(helper, llm):
    path = f"results/facts_{a.tag}_{helper.replace(':', '-')}.json"
    done = load(path)
    t0 = time.time(); n = 0
    for c in ids:
        by = {}
        for t in convs[c].turns: by.setdefault(t["session"], []).append(t)
        for ses, ts in by.items():
            key = f"{c}:{ses}"
            if key in done: continue
            d = ts[0]["date"]
            chat = f"[{d}]\n" + "\n".join(f"{t['id']} {t['speaker']}: {E.with_dates(t['text'], d)}" + (f" [image: {t['cap']}]" if t["cap"] else "") for t in ts)
            s0 = time.time()
            r = llm(FACT_PROMPT.format(chat=chat), max_tokens=700)[0]
            f = parse_facts(r, ses, {t["id"]: f"{t['speaker']}: {E.with_dates(t['text'], d)}" + (f" {t['cap']}" if t["cap"] else "") for t in ts})
            done[key] = dict(facts=f, secs=round(time.time() - s0, 1), raw=r[:2500])
            json.dump(done, open(path, "w")); n += 1
            print(f"fact file {helper} {key} {len(f)} facts", flush=True)
    print(f"facts ready for {helper}: chats {len(done)}, empty {sum(not v['facts'] for v in done.values())}, facts {sum(len(v['facts']) for v in done.values())}, links corrected {sum(x['moved'] for v in done.values() for x in v['facts'])}, seconds per chat {round(np.mean([v['secs'] for v in done.values()]), 1)}", flush=True)
    return done


class FactIndex:
    def __init__(s, c, facts):
        s.items = [(x["fact"], [REF[c][t] for t in x["turns"]]) for k, v in facts.items() if k.startswith(f"{c}:") for x in v["facts"]]
        s.search = MEM[c].search
        s.w = [E.words(f) for f, _ in s.items]
        s.df = Counter(w for ws in s.w for w in set(ws))
        s.avg = sum(map(len, s.w)) / max(1, len(s.w))
        s.v = s.search._embed([f for f, _ in s.items]) if s.items else None

    def find(s, q):
        if not s.items: return []
        qv, qw, N = s.search._embed([q])[0], set(E.words(q)), len(s.items)
        mid, st = s.search._curve(q)
        sc = s.v @ qv
        for i, ws in enumerate(s.w):
            b = 0.
            for w in qw & set(ws):
                tf = ws.count(w); b += math.log(1 + (N - s.df[w] + .5) / (s.df[w] + .5)) * tf * 2.2 / (tf + 1.2 * (.5 + .5 * len(ws) / s.avg))
            if b > 0: sc[i] += 1 / (1 + math.exp(-st * (b - mid)))
        return np.argsort(-sc).tolist()


def clean_cues(r):
    out = []
    for line in r.splitlines():
        line = re.sub(r"^[\s\-\*\d\.\)]+", "", line).strip().strip('"')
        if 2 < len(line) < 120: out.append(line)
    return out[:3]


# ---------- building the text the model reads ----------
def select(c, order, budget, facts=()):
    m, out, seen, used, lines = MEM[c], [], set(), 0, []
    for f in facts:
        t = E.count_tokens(f) + 2
        if used + t > budget * .3: break
        lines.append("[Noted fact] " + f); used += t

    def take(sid):
        nonlocal used
        if sid in seen: return
        seen.add(sid); out.append(sid); used += m._tokens(sid) + 8
    for sid in order:
        if used >= budget: break
        take(sid)
        nxt = m._next_in_chat(sid)
        if nxt is not None and used < budget: take(nxt)
    return out, used, "\n".join(lines + ([m.text_of(sorted(out))] if out else []))


plan = [(c, i) for c, i in K.balanced(convs, ids, a.sample) if convs[c].qs[i]["cat"] != 5]
plan = [(c, i) for c, i in plan if any(e in REF[c] for e in convs[c].qs[i]["ev"])]
print(len(plan), "questions", flush=True)
answer_llm = K.LLM(a.url, a.answer_model, "local", 1200)
path = f"results/helper_{a.tag}.jsonl"
done = {(r["conv"], r["i"], r["system"]) for r in map(json.loads, open(path))} if os.path.exists(path) else set()


def record(out, c, i, system, sids, used, text, cue_list=None):
    q = convs[c].qs[i]
    gold = {REF[c][e] for e in q["ev"] if e in REF[c]}
    pred, _, secs = answer_llm(K.ANSWER.format(ctx=text or "(no excerpts)", q=q["q"]), max_tokens=60)
    out.write(json.dumps(dict(conv=c, i=i, system=system, cat=K.CATS[q["cat"]], q=q["q"], gold=q["answer"], pred=pred, recall=len(set(sids) & gold) / len(gold), tokens=used, secs=secs, cues=cue_list)) + "\n")
    out.flush()


with open(path, "a") as out:
    for n, (c, i) in enumerate(plan, 1):
        q = convs[c].qs[i]["q"]
        s = MEM[c].search
        for name in ("current", "plain", "cabinet", "network"):
            if (c, i, name) in done: continue
            if name == "current":
                sids = CUR[c].find(q)[0]
                u = sum(CUR[c]._tokens(x) + 8 for x in sids)
                record(out, c, i, name, sids, u, CUR[c].text_of(sids))
            else:
                order = net_order(c, q) if name == "network" else s.order(q, name)
                sids, u, txt = select(c, order, a.budget)
                record(out, c, i, name, sids, u, txt)
        if n % 10 == 0: print(f"question {n}/{len(plan)} base", flush=True)
print("base systems done", flush=True)

for helper in a.helpers.split(","):
    helper = helper.strip()
    tag = helper.replace(":", "-")
    llm = K.LLM(a.url, helper, "local", 1200)
    try:
        llm("Reply with one word.", max_tokens=5)
    except Exception as e:
        print("helper not available, skipping:", helper, str(e)[:100], flush=True)
        continue
    facts = write_facts(helper, llm)
    FI = {c: FactIndex(c, facts) for c in ids}
    cue_path = f"results/cues_{a.tag}_{tag}.json"
    cues = load(cue_path)
    with open(path, "a") as out:
        for n, (c, i) in enumerate(plan, 1):
            q = convs[c].qs[i]["q"]
            s = MEM[c].search
            key = f"{c}:{i}"
            if key not in cues:
                r, _, secs = llm(CUE_PROMPT.format(q=q), max_tokens=60)
                cues[key] = dict(cues=clean_cues(r), secs=round(secs, 2))
                json.dump(cues, open(cue_path, "w"))
            net = net_order(c, q)
            fx = FI[c]
            hits = fx.find(q)
            fact_sids = [t for h in hits for t in fx.items[h][1]]
            fact_text = [fx.items[h][0] for h in hits[:8]]
            cue_orders = [s.order(x, "plain") for x in cues[key]["cues"]]
            for name, order, shown in ((f"facts|{helper}", fuse([net, fact_sids]), fact_text), (f"cues|{helper}", fuse([net] + cue_orders), ()), (f"both|{helper}", fuse([net, fact_sids] + cue_orders), fact_text)):
                if (c, i, name) in done: continue
                sids, u, txt = select(c, order, a.budget, shown)
                record(out, c, i, name, sids, u, txt, cues[key]["cues"])
            if n % 10 == 0: print(f"question {n}/{len(plan)} helper {helper}", flush=True)
    print("helper done", helper, flush=True)
print("Done", flush=True)

In [ ]:
import glob, os, shutil, subprocess, sys
os.chdir(f"{WORK}/Engram")
os.makedirs("results", exist_ok=True)
for f in glob.glob(f"{SAVE}/*{TAG}*"): shutil.copy(f, "results")
def save():
    for f in glob.glob(f"results/*_{TAG}*"): shutil.copy(f, SAVE)
run = subprocess.Popen([sys.executable, "-u", "helper_test.py", "--url", "http://localhost:11434/v1", "--answer_model", ANSWER_MODEL, "--helpers", ",".join(HELPERS), "--tag", TAG], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
n = 0
for line in run.stdout:
    if not line.startswith("fact file"): print(line, end="")
    if line.startswith(("question ", "fact file", "facts ready", "helper done", "base systems")):
        n += 1
        if n % 5 == 0 or not line.startswith(("question ", "fact file")): save()
code = run.wait()
save()
print("Finished" if code == 0 else f"Stopped with an error, code {code}. Run this cell again to continue.")
print(sorted(os.listdir(SAVE)))

When it says Finished, or if the compute runs out, send Claude the files named `helper_dev.jsonl`, `facts_dev_...json` and `cues_dev_...json` from the `engram_results` folder. On Kaggle they are in the Output panel on the right.